# ARC AGI 3 Single-Game Colab Training

This notebook is for a focused experiment on **one game at a time**.

1. Read the latest project code from `MyDrive/ARC Prize 2026 - ARC-AGI-3`
2. Copy that code to the Colab local disk
3. Read the full human training `.gz` from `MyDrive/ARC2026_AGI_3/Input_Data`
4. Filter to one selected game inside `src.train`
5. Split that game by **episode** into train/val holdouts
6. Train on Colab local disk
7. Evaluate the trained checkpoint on that same game with real rollout


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

PROJECT_SOURCE_MODE = 'drive_dir'  # Default and recommended. Use 'drive_zip' only if you uploaded a project bundle zip.
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3')
DRIVE_PROJECT_ZIP = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3/Local_Output/Colab_Bundles/arc_agi3_colab_bundle.zip')
DRIVE_INPUT_DATA_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Input_Data')
DRIVE_OUTPUT_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Training_Output')
LOCAL_WORKDIR = Path('/content/ARC Prize 2026 - ARC-AGI-3')
LOCAL_INPUT_DATA_BASE = Path('/content/ARC2026_AGI_3_Input_Data')
RUN_TS = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_INPUT_DATA_BASE.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print('Project source mode:', PROJECT_SOURCE_MODE)
print('Drive project root:', DRIVE_PROJECT_ROOT)
print('Drive project zip:', DRIVE_PROJECT_ZIP)
print('Drive input data base:', DRIVE_INPUT_DATA_BASE)
print('Drive output base:', DRIVE_OUTPUT_BASE)
print('Local workdir:', LOCAL_WORKDIR)
print('Local input data base:', LOCAL_INPUT_DATA_BASE)
print('Output root:', OUTPUT_ROOT)


In [ ]:
import shutil

if LOCAL_WORKDIR.exists():
    shutil.rmtree(LOCAL_WORKDIR)
LOCAL_WORKDIR.parent.mkdir(parents=True, exist_ok=True)

if PROJECT_SOURCE_MODE == 'drive_dir':
    get_ipython().system('rsync -a --delete --exclude .git --exclude .venv --exclude __pycache__ --exclude .DS_Store --exclude Local_Output --exclude Gif_Demo --exclude OpenLab_Backup_* "{}"/ "{}"/'.format(DRIVE_PROJECT_ROOT, LOCAL_WORKDIR))
elif PROJECT_SOURCE_MODE == 'drive_zip':
    if not DRIVE_PROJECT_ZIP.exists():
        raise FileNotFoundError(f'Project zip not found: {DRIVE_PROJECT_ZIP}')
    get_ipython().system('unzip -q "{}" -d /content'.format(DRIVE_PROJECT_ZIP))
else:
    raise ValueError(f'Unsupported PROJECT_SOURCE_MODE: {PROJECT_SOURCE_MODE}')

get_ipython().run_line_magic('cd', str(LOCAL_WORKDIR))


In [ ]:
import importlib.util, subprocess
from pathlib import Path

def run(cmd):
    print('>>>', cmd)
    subprocess.check_call(cmd, shell=True)

def run_streaming(cmd, log_path=None):
    print('>>>', cmd)
    handle = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        handle = log_path.open('a', encoding='utf-8')
    try:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            if handle is not None:
                handle.write(line)
                handle.flush()
        return_code = proc.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)
    finally:
        if handle is not None:
            handle.close()

def has_module(name):
    return importlib.util.find_spec(name) is not None

run('python -m pip install -U pip wheel setuptools')

if has_module('torch'):
    print('torch already available, skipping torch/vision/audio install')
else:
    run('python -m pip install -U torch torchvision torchaudio')

if has_module('arc_agi') and has_module('arcengine'):
    print('arc_agi and arcengine already available, skipping install')
else:
    try:
        run('python -m pip install -U arc-agi==0.9.8 arcengine==0.9.3')
    except Exception:
        print('PyPI install failed, trying local wheels...')
        run('python -m pip install arc_agi_3_wheels/*.whl')

run('python - <<\'PY\'\nimport torch\nprint("torch", torch.__version__)\nprint("cuda", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("device", torch.cuda.get_device_name(0))\nPY')


In [ ]:
SINGLE_GAME_ID = 'ar25'  # Change this to ar25, ls20, r11l, lp85, etc.
EPISODE_VAL_FRACTION = 0.2
RUN_TAG = f'single_game_{SINGLE_GAME_ID}'
OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_TAG / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

HARDWARE_PROFILE = 'a100'
CHECKPOINT_EVERY_STEPS = 100
LOG_EVERY_BATCHES = 5
DATA_WORKERS = 6
USE_LOCAL_DATA_CACHE = True
RESUME_CHECKPOINT = None

SOURCE_HUMAN_GZ = DRIVE_INPUT_DATA_BASE / 'arc_agi_3_public_demo_human_testing.gz'
if not SOURCE_HUMAN_GZ.exists():
    raise FileNotFoundError(f'Human training gzip not found: {SOURCE_HUMAN_GZ}')

LOCAL_INPUT_DATA_BASE.mkdir(parents=True, exist_ok=True)
LOCAL_SINGLE_GAME_GZ = LOCAL_INPUT_DATA_BASE / f'single_game_human_{SINGLE_GAME_ID}.episodes.jsonl.gz'
print('Single game:', SINGLE_GAME_ID)
print('Episode val fraction:', EPISODE_VAL_FRACTION)
print('Output root:', OUTPUT_ROOT)
print('Source human data:', SOURCE_HUMAN_GZ)
print('Single-game data path:', LOCAL_SINGLE_GAME_GZ)


In [ ]:
filter_cmd = f'''python -m src.filter_episodes \
  --input "{SOURCE_HUMAN_GZ}" \
  --output "{LOCAL_SINGLE_GAME_GZ}" \
  --games "{SINGLE_GAME_ID}"'''

run_streaming(filter_cmd, OUTPUT_ROOT / 'filter_stdout.log')

resume_arg = '' if RESUME_CHECKPOINT is None else f' --resume "{RESUME_CHECKPOINT}"'

train_cmd = f'''PYTHONUNBUFFERED=1 python -m src.train \
  --project-root "{LOCAL_WORKDIR}" \
  --data "{LOCAL_SINGLE_GAME_GZ}" \
  --games "{SINGLE_GAME_ID}" \
  --split-mode episode \
  --episode-val-fraction {EPISODE_VAL_FRACTION} \
  --output-dir "{OUTPUT_ROOT}" \
  --hardware-profile {HARDWARE_PROFILE} \
  --max-steps 192 \
  --online-val-games 1 \
  --checkpoint-every-steps {CHECKPOINT_EVERY_STEPS} \
  --data-workers {DATA_WORKERS} \
  --log-every-batches {LOG_EVERY_BATCHES}{resume_arg}'''

run_streaming(train_cmd, OUTPUT_ROOT / 'train_stdout.log')


In [ ]:
eval_best_cmd = f'''python -m src.evaluate \
  --project-root "{LOCAL_WORKDIR}" \
  --checkpoint "{OUTPUT_ROOT / 'checkpoints' / 'best.pth'}" \
  --output "{OUTPUT_ROOT / f'{SINGLE_GAME_ID}_best_public_eval.json'}" \
  --games "{SINGLE_GAME_ID}"'''

run_streaming(eval_best_cmd, OUTPUT_ROOT / 'eval_best_stdout.log')

eval_last_cmd = f'''python -m src.evaluate \
  --project-root "{LOCAL_WORKDIR}" \
  --checkpoint "{OUTPUT_ROOT / 'checkpoints' / 'last.pth'}" \
  --output "{OUTPUT_ROOT / f'{SINGLE_GAME_ID}_last_public_eval.json'}" \
  --games "{SINGLE_GAME_ID}"'''

run_streaming(eval_last_cmd, OUTPUT_ROOT / 'eval_last_stdout.log')


In [ ]:
import json
import pandas as pd

metrics_path = OUTPUT_ROOT / 'metrics.csv'
display(pd.read_csv(metrics_path).tail())

def show_eval(path):
    payload = json.loads(path.read_text())
    print(path.name, 'mean_score=', payload['mean_score'], 'mean_levels_completed=', payload['mean_levels_completed'])
    display(pd.DataFrame(payload['games']).sort_values(['score', 'levels_completed', 'actions_taken'], ascending=[False, False, True]))

print('Best checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'best.pth')
print('Last checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'last.pth')
show_eval(OUTPUT_ROOT / f'{SINGLE_GAME_ID}_best_public_eval.json')
show_eval(OUTPUT_ROOT / f'{SINGLE_GAME_ID}_last_public_eval.json')
